# 1\.1\.4 Load Bus Speeds

This notebook loads and validates the MTA Bus Route Segment Speeds datasets spanning 2023–2025\. During inspection, the raw files proved too large for a simple in\-memory dataframe workflow, so the notebook shifts to a chunked parquet staging approach to make the dataset manageable and reproducible\. The rest of the notebook validates the staged outputs, checks data quality, clarifies the dataset's observation grain, and confirms that the core bus mobility metrics are usable for downstream spatial alignment and multimodal analysis\.

In [1]:
from pathlib import Path
import requests
import glob

import pyarrow.parquet as pq
import pandas as pd
from collections import Counter

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1\.1\.4\.1 Load the bus speeds dataset

The bus speed datasets give us a very different lens into NYC mobility than ridership alone\. Instead of simply counting passengers, these datasets measure how efficiently buses are actually moving through the city street network — segment by segment, route by route, and hour by hour\.

For congestion pricing analysis, this is probably one of the more interesting transit datasets in the entire project\. If roadway congestion changes after the tolling rollout, bus speeds and travel times should theoretically react pretty quickly, especially in Manhattan and other dense corridor\-heavy parts of the city\.

The datasets below contain roadway\-segment\-level measurements like average bus speed, travel time, trip counts, route metadata, and spatial coordinates that we can later integrate into the broader multimodal mobility pipeline\.

Also: the normal browser export workflow basically melted down trying to download 10\+ million rows, so we are pulling the datasets programmatically through the Socrata API instead\. Much cleaner, much more reproducible, and honestly much more appropriate for a project at this scale\.

In [2]:
# ---------------------------------------------------------------------
# Download MTA bus route segment speed data via Socrata API
# ---------------------------------------------------------------------

SOURCE_DATA_DIR = Path("source_data")
BUS_DATA_DIR = SOURCE_DATA_DIR / "bus"
BUS_DATA_DIR.mkdir(parents=True, exist_ok=True)

bus_downloads = {
    "mta_bus_route_segment_speeds_2023_2024.csv":
        "https://data.ny.gov/api/views/58t6-89vi/rows.csv?accessType=DOWNLOAD",
    "mta_bus_route_segment_speeds_2025_plus.csv":
        "https://data.ny.gov/api/views/kufs-yh3x/rows.csv?accessType=DOWNLOAD",
}

for filename, url in bus_downloads.items():
    output_path = BUS_DATA_DIR / filename

    if output_path.exists() and output_path.stat().st_size > 0:
        size_mb = output_path.stat().st_size / (1024 * 1024)
        print(f"Already downloaded, skipping: {filename} ({size_mb:.1f} MB)")
        continue

    print(f"Downloading {filename}...")

    response = requests.get(
        url,
        stream=True,
        timeout=300,
    )

    response.raise_for_status()

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"Saved bus speeds dataset to: {output_path} ({size_mb:.1f} MB)")

Already downloaded, skipping: mta_bus_route_segment_speeds_2023_2024.csv (2851.3 MB)
Already downloaded, skipping: mta_bus_route_segment_speeds_2025_plus.csv (1793.5 MB)


Since the datasets are so huge, let's be selective about the columns we will use\. Let's pull a sample from each bus dataset and analyze\.\.\.

In [3]:
BUS_2023_2024_PATH = BUS_DATA_DIR / "mta_bus_route_segment_speeds_2023_2024.csv"
BUS_2025_PLUS_PATH = BUS_DATA_DIR / "mta_bus_route_segment_speeds_2025_plus.csv"

bus_2023_2024_sample = pd.read_csv(BUS_2023_2024_PATH, nrows=1000)
bus_2025_plus_sample = pd.read_csv(BUS_2025_PLUS_PATH, nrows=1000)

print("Infos")
print(bus_2023_2024_sample.info())
print(bus_2025_plus_sample.info())

print("Check column consistency")
print(set(bus_2023_2024_sample.columns) == set(bus_2025_plus_sample.columns))



Infos
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 24 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Year                              1000 non-null   int64  
 1   Month                             1000 non-null   int64  
 2   Timestamp                         1000 non-null   object 
 3   Day of Week                       1000 non-null   object 
 4   Hour of Day                       1000 non-null   int64  
 5   Route ID                          1000 non-null   object 
 6   Direction                         1000 non-null   object 
 7   Borough                           1000 non-null   object 
 8   Route Type                        1000 non-null   object 
 9   Stop Order                        1000 non-null   int64  
 10  Timepoint Stop ID                 1000 non-null   int64  
 11  Timepoint Stop Name               1000 non-null   object 
 12  T

In [4]:
bus_2023_2024_sample.head()

,Year,Month,Timestamp,Day of Week,Hour of Day,Route ID,Direction,Borough,Route Type,Stop Order,Timepoint Stop ID,Timepoint Stop Name,Timepoint Stop Latitude,Timepoint Stop Longitude,Next Timepoint Stop ID,Next Timepoint Stop Name,Next Timepoint Stop Latitude,Next Timepoint Stop Longitude,Road Distance,Average Travel Time,Average Road Speed,Bus Trip Count,Timepoint Stop Georeference,Next Timepoint Stop Georeference
0,2023,3,03/01/2023 09:00:00 AM,Friday,9,B6,W,Brooklyn,Limited,1,306921,LIVONIA AV/ASHFORD ST,40.666382,-73.883617,300590,COZINE AV /ASHFORD ST,40.658352,-73.877775,0.724,5.334420,8.143341,46,POINT (-73.883617 40.666382),POINT (-73.877775 40.658352)
1,2023,3,03/01/2023 01:00:00 PM,Friday,13,B6,W,Brooklyn,Limited,1,306921,LIVONIA AV/ASHFORD ST,40.666382,-73.883617,300590,COZINE AV /ASHFORD ST,40.658352,-73.877775,0.724,5.677776,7.650882,36,POINT (-73.883617 40.666382),POINT (-73.877775 40.658352)
2,2023,3,03/01/2023 01:00:00 AM,Friday,1,BX42,E,Bronx,Local,29,102613,RANDALL AV/E TREMONT AV,40.826154,-73.821999,104299,HARDING AV/HOSMER AV,40.813235,-73.826535,2.021,10.143750,11.954159,8,POINT (-73.821999 40.826154),POINT (-73.826535 40.813235)
3,2023,3,03/01/2023 01:00:00 PM,Friday,13,BXM1,N,Bronx,Express,13,404250,BROADWAY/W 207 ST,40.867705,-73.920997,984006,W 230 ST / BROADWAY,40.877199,-73.906891,1.027,7.516680,8.197768,5,POINT (-73.920997 40.867705),POINT (-73.906891 40.877199)
4,2023,3,03/01/2023 12:00:00 PM,Friday,12,BXM1,S,Bronx,Express,29,403424,LEXINGTON AV/E 50 ST,40.756659,-73.972382,403821,LEXINGTON AV / E 34 ST,40.746315,-73.979970,0.817,8.238894,5.949828,9,POINT (-73.972382 40.756659),POINT (-73.97997 40.746315)


Data Dictionary

Year\. Calendar year associated with the roadway segment observation\.
Month\. Calendar month associated with the roadway segment observation\.
Timestamp\. Timestamp representing the hour\-level observation window for the roadway segment metrics\.
Day of Week\. Day\-of\-week label corresponding to the timestamped observation period\.
Hour of Day\. Hour extracted from the timestamp, representing the hourly roadway measurement interval\.
Route ID\. MTA bus route identifier \(e\.g\., B6, BX42, Q58, M15\)\.
Direction\. Direction of travel for the bus route segment \(e\.g\., N/S/E/W\)\.
Borough\. NYC borough associated with the roadway segment observation\.
Route Type\. Bus service classification, such as Local, Limited, or Express\.
Stop Order\. Sequential stop position along the bus route pattern\. Lower values generally represent earlier segments in the route\.
Timepoint Stop ID\. Unique identifier for the starting timepoint stop of the roadway segment\.
Timepoint Stop Name\. Human\-readable name of the starting timepoint stop\.
Timepoint Stop Latitude\. Latitude coordinate of the starting timepoint stop\.
Timepoint Stop Longitude\. Longitude coordinate of the starting timepoint stop\.
Next Timepoint Stop ID\. Unique identifier for the downstream/end timepoint stop of the roadway segment\.
Next Timepoint Stop Name\. Human\-readable name of the downstream/end timepoint stop\.
Next Timepoint Stop Latitude\. Latitude coordinate of the downstream/end timepoint stop\.
Next Timepoint Stop Longitude\. Longitude coordinate of the downstream/end timepoint stop\.
Road Distance\. Estimated roadway distance traveled between the two timepoint stops, typically measured in miles\.
Average Travel Time\. Average observed bus travel time between the two timepoint stops during the observation window\.
Average Road Speed\. Average observed roadway speed for buses traveling across the roadway segment during the observation window\.
Bus Trip Count\. Number of bus trips contributing to the aggregated roadway segment statistics for the observation window\.
Timepoint Stop Georeference\. Geospatial POINT representation of the starting timepoint stop location\.
Next Timepoint Stop Georeference\. Geospatial POINT representation of the downstream/end timepoint stop location\.

Findings: The bus segment speed datasets honestly look a lot richer than I expected\. This is not just a ridership table — it is effectively a roadway\-level mobility efficiency dataset for the NYC bus network\. The data captures how efficiently buses are moving through the city street grid across specific roadway segments, routes, directions, hours, and boroughs\.

At this stage, it probably makes more sense to preserve nearly the entire schema during staging rather than aggressively dropping columns too early\. Metrics like Average Road Speed, Average Travel Time, and Bus Trip Count are obvious candidates for downstream congestion\-outlier detection, time\-series decomposition, forecasting, and potential mobility\-regime clustering\. Route metadata such as Route ID, Route Type, Direction, and even Stop Order may also become useful later if we want to distinguish structurally different transit corridors or identify recurring congestion patterns along specific routes\.

The geographic columns are especially valuable because they give us actual roadway\-segment endpoints that can later be spatially linked into Taxi Zones and other mobility geographies used throughout the project\. In particular, fields like Timepoint Stop Latitude, Timepoint Stop Longitude, Next Timepoint Stop Latitude, Next Timepoint Stop Longitude, and the corresponding georeference POINT fields give us enough spatial detail to eventually build segment geometries and integrate bus\-mobility signals with roadway traffic counts, taxi demand, and potentially even social\-media discussions around congestion pricing and perceived mobility disruption\.

## 1\.1\.4\.2 Convert raw bus \.csv files into staged parquet partitions

The raw MTA bus segment speed datasets are honestly massive — the 2023–2024 CSV alone is several gigabytes and contains millions of roadway\-segment observations\. Trying to repeatedly load datasets at this scale directly into pandas is not especially practical and started putting noticeable pressure on the notebook environment during exploratory work\.

In [5]:
for path in BUS_DATA_DIR.glob("*.csv"):
    size_mb = path.stat().st_size / (1024 * 1024)
    size_gb = size_mb / 1024

    print(
        f"{path.name}: "
        f"{size_mb:,.1f} MB "
        f"({size_gb:.2f} GB)"
    )

mta_bus_route_segment_speeds_2023_2024.csv: 2,851.3 MB (2.78 GB)
mta_bus_route_segment_speeds_2025_plus.csv: 1,793.5 MB (1.75 GB)


So before doing deeper QA and inspection, the raw CSV files will first be converted into staged parquet partitions that are much easier to work with incrementally\. Just as a reminder: parquet is a compressed columnar storage format that is substantially faster and more memory\-efficient than raw CSV files for analytical workflows like filtering, aggregation, geospatial processing, and time\-series analysis\.

Importantly, this does not prevent us from inspecting or validating the datasets\. The staged parquet partitions can still be sampled, profiled, queried, and QA’d incrementally without needing to load the full raw datasets into memory all at once\.

In [6]:
# ---------------------------------------------------------------------
# Configure staged bus parquet output
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

BUS_PARQUET_DIR = (
    PIPELINE_DATA_DIR / "1.1.4 bus_route_segment_speeds_parquet"
)

BUS_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

print(f"Bus parquet output directory: {BUS_PARQUET_DIR}")

Bus parquet output directory: pipeline_data/1.1.4 bus_route_segment_speeds_parquet


In [7]:
# ---------------------------------------------------------------------
# Convert raw bus CSV files into staged parquet partitions
# ---------------------------------------------------------------------
# Bus speed files are too large for routine full-CSV loading. This helper
# converts them once into chunked parquet files, parsing Timestamp during the
# conversion so downstream QA can read smaller typed partitions instead of
# repeatedly reparsing multi-GB CSVs.

def csv_to_parquet_partitions(
    csv_path,
    output_dir,
    dataset_label,
    chunksize=500_000,
    overwrite=False,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    existing_parts = sorted(
        output_dir.glob(f"{dataset_label}_part_*.parquet")
    )

    if existing_parts and not overwrite:
        print(
            f"Skipping {dataset_label}: "
            f"{len(existing_parts)} parquet partitions already exist."
        )
        return

    if existing_parts and overwrite:
        print(f"Overwriting existing partitions for {dataset_label}...")
        for part in existing_parts:
            part.unlink()

    for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunksize)):
        print(f"Processing {dataset_label} chunk {i + 1:,}...")

        chunk["Timestamp"] = pd.to_datetime(
            chunk["Timestamp"],
            format="%m/%d/%Y %I:%M:%S %p",
            errors="coerce"
        )       
        
        output_path = output_dir / f"{dataset_label}_part_{i:04d}.parquet"

        chunk.to_parquet(
            output_path,
            index=False
        )

    print(f"Finished converting {dataset_label}")

In [8]:
# ---------------------------------------------------------------------
# Generate staged parquet partitions from raw bus CSV files
# ---------------------------------------------------------------------

csv_to_parquet_partitions(
    csv_path=BUS_2023_2024_PATH,
    output_dir=BUS_PARQUET_DIR,
    dataset_label="bus_2023_2024",
    chunksize=500_000,
    overwrite=False,
)

csv_to_parquet_partitions(
    csv_path=BUS_2025_PLUS_PATH,
    output_dir=BUS_PARQUET_DIR,
    dataset_label="bus_2025_plus",
    chunksize=500_000,
    overwrite=False,
)

Skipping bus_2023_2024: 24 parquet partitions already exist.
Skipping bus_2025_plus: 15 parquet partitions already exist.


## 1\.1\.4\.3 Bus parquet chunk inspection

Now that the parquet partitioning is done, let's inspect the files\. Because these parquet QA checks are expensive to recompute, the notebook caches partition\-level and timestamp QA outputs as reusable manifest files under \`pipeline\_data/qa/\`\. These manifests allow the notebook to reload prior QA results quickly after kernel resets, while preserving the option to regenerate QA by setting \`REBUILD\_QA = True\` if the staged parquet files change\.

In [9]:
# ---------------------------------------------------------------------
# Configure cached QA manifest
# ---------------------------------------------------------------------

QA_DIR = PIPELINE_DATA_DIR / "qa"
QA_DIR.mkdir(parents=True, exist_ok=True)

BUS_PARTITION_QA_MANIFEST = QA_DIR / "bus_parquet_partition_qa.parquet"

# Set to True only when the staged parquet files have changed
REBUILD_QA = False

In [10]:
# ---------------------------------------------------------------------
# Inspect staged bus parquet partitions, using cached QA manifest
# ---------------------------------------------------------------------

if BUS_PARTITION_QA_MANIFEST.exists() and not REBUILD_QA:
    partition_health_df = pd.read_parquet(BUS_PARTITION_QA_MANIFEST)
    print(f"Loaded cached QA manifest: {BUS_PARTITION_QA_MANIFEST}")

else:
    partition_health = []

    for path in sorted(BUS_PARQUET_DIR.glob("*.parquet")):
        pf = pq.ParquetFile(path)

        partition_health.append({
            "file": path.name,
            "size_mb": path.stat().st_size / (1024 * 1024),
            "num_rows": pf.metadata.num_rows,
            "num_columns": pf.metadata.num_columns,
        })

    partition_health_df = pd.DataFrame(partition_health)
    partition_health_df.to_parquet(BUS_PARTITION_QA_MANIFEST, index=False)

    print(f"Rebuilt QA manifest: {BUS_PARTITION_QA_MANIFEST}")

print("Partition count:", len(partition_health_df))
print("Total rows:", partition_health_df["num_rows"].sum())
print("Total size GB:", partition_health_df["size_mb"].sum() / 1024)

partition_health_df

Loaded cached QA manifest: pipeline_data/qa/bus_parquet_partition_qa.parquet
Partition count: 39
Total rows: 18937024
Total size GB: 0.5399150345474482


,file,size_mb,num_rows,num_columns
0,bus_2023_2024_part_0000.parquet,14.177539,500000,24
1,bus_2023_2024_part_0001.parquet,15.045886,500000,24
2,bus_2023_2024_part_0002.parquet,14.899799,500000,24
3,bus_2023_2024_part_0003.parquet,14.983352,500000,24
4,bus_2023_2024_part_0004.parquet,14.925962,500000,24
5,bus_2023_2024_part_0005.parquet,14.999928,500000,24
6,bus_2023_2024_part_0006.parquet,15.133587,500000,24
7,bus_2023_2024_part_0007.parquet,15.020267,500000,24
8,bus_2023_2024_part_0008.parquet,14.895361,500000,24
9,bus_2023_2024_part_0009.parquet,15.080462,500000,24


Findings\. The staged bus dataset contains 39 parquet partitions and 18\.9 million route\-segment observations\. All partitions contain the expected 24\-column schema, and partition sizes align with the intended 500,000\-row chunking strategy\. Together, these results suggest the parquet staging process completed successfully without obvious row\-loss, schema, or partition\-integrity issues\.

Since we ran into some initial issues with timestamp conversions while creating the partitioned parquet files, let's do a sanity check to make sure they're all okay\.\.\.

In [11]:
# ---------------------------------------------------------------------
# Validate timestamp parsing, using cached QA summary
# ---------------------------------------------------------------------

BUS_TIMESTAMP_QA_MANIFEST = QA_DIR / "bus_timestamp_qa.parquet"

if BUS_TIMESTAMP_QA_MANIFEST.exists() and not REBUILD_QA:
    timestamp_qa_df = pd.read_parquet(BUS_TIMESTAMP_QA_MANIFEST)
    print(f"Loaded cached timestamp QA: {BUS_TIMESTAMP_QA_MANIFEST}")

else:
    timestamp_nulls = 0
    row_count = 0

    for path in sorted(BUS_PARQUET_DIR.glob("*.parquet")):
        temp = pd.read_parquet(path, columns=["Timestamp"])
        timestamp_nulls += temp["Timestamp"].isna().sum()
        row_count += len(temp)

    timestamp_qa_df = pd.DataFrame([{
        "rows_checked": row_count,
        "timestamp_nulls": timestamp_nulls,
        "timestamp_null_rate": timestamp_nulls / row_count,
    }])

    timestamp_qa_df.to_parquet(BUS_TIMESTAMP_QA_MANIFEST, index=False)
    print(f"Rebuilt timestamp QA: {BUS_TIMESTAMP_QA_MANIFEST}")

timestamp_qa_df

Loaded cached timestamp QA: pipeline_data/qa/bus_timestamp_qa.parquet


,rows_checked,timestamp_nulls,timestamp_null_rate
0,18937024,0,0.0


Findings: Timestamp parsing was successful, with a 0\.0% null rate across all 18\.9 million observations\. This resolves the earlier parsing concerns and clears the dataset for temporal analysis\.

## 1\.1\.4\.4 Bus data inspection

We will go through our usual dataset inspections: datatype correctness and consistency, missingness, categorical distributions and duplicates\.

In [12]:
# ---------------------------------------------------------------------
# Inspect datatype of each column and consistency across Parquet files
# Uses cached QA manifest unless REBUILD_QA = True
# ---------------------------------------------------------------------

BUS_SCHEMA_QA_MANIFEST = QA_DIR / "bus_schema_qa.parquet"

if BUS_SCHEMA_QA_MANIFEST.exists() and not REBUILD_QA:
    schema_qa_df = pd.read_parquet(BUS_SCHEMA_QA_MANIFEST)
    print(f"Loaded cached schema QA: {BUS_SCHEMA_QA_MANIFEST}")

else:
    parquet_chunks = sorted(BUS_PARQUET_DIR.glob("*.parquet"))

    schema_records = []

    print(f"Opening footers for {len(parquet_chunks)} Parquet chunks...")

    for chunk_file in parquet_chunks:
        schema = pq.read_schema(chunk_file)

        for field in schema:
            schema_records.append({
                "file": chunk_file.name,
                "column_name": field.name,
                "discovered_datatype": str(field.type),
            })

    schema_qa_df = pd.DataFrame(schema_records)

    schema_qa_df.to_parquet(BUS_SCHEMA_QA_MANIFEST, index=False)
    print(f"Rebuilt schema QA: {BUS_SCHEMA_QA_MANIFEST}")

# Build summary from cached or rebuilt detail table
schema_summary_df = (
    schema_qa_df
    .groupby("column_name", as_index=False)
    .agg(
        discovered_datatypes=("discovered_datatype", lambda x: sorted(set(x))),
        num_discovered_datatypes=("discovered_datatype", lambda x: len(set(x))),
        files_seen=("file", "nunique"),
    )
)

all_files_valid = (schema_summary_df["num_discovered_datatypes"] == 1).all()

print("\n" + "="*50)
print("        PARQUET CHUNK SCHEMA INSPECTION MAP")
print("="*50)
print(f"{'COLUMN NAME':<35} | {'DISCOVERED DATATYPE(S)'}")
print("-"*50)

for _, row in schema_summary_df.iterrows():
    col_name = row["column_name"]
    types_list = row["discovered_datatypes"]

    if len(types_list) > 1:
        print(f"❌ {col_name:<33} | CRITICAL MISMATCH: {types_list}")
    else:
        print(f"   {col_name:<33} | {types_list[0]}")

print("="*50)

if all_files_valid:
    print(f"\n🎉 SCHEMA VALIDATION PASSED: All {schema_qa_df['file'].nunique()} chunks share a uniform schema!")
else:
    print("\n⚠️ SCHEMA VALIDATION FAILED: Type contradictions exist across parquet chunks.")

display(schema_summary_df)

Loaded cached schema QA: pipeline_data/qa/bus_schema_qa.parquet

        PARQUET CHUNK SCHEMA INSPECTION MAP
COLUMN NAME                         | DISCOVERED DATATYPE(S)
--------------------------------------------------
   Average Road Speed                | double
   Average Travel Time               | double
   Borough                           | string
   Bus Trip Count                    | int64
   Day of Week                       | string
   Direction                         | string
   Hour of Day                       | int64
   Month                             | int64
   Next Timepoint Stop Georeference  | string
   Next Timepoint Stop ID            | int64
   Next Timepoint Stop Latitude      | double
   Next Timepoint Stop Longitude     | double
   Next Timepoint Stop Name          | string
   Road Distance                     | double
   Route ID                          | string
   Route Type                        | string
   Stop Order                        | int64
  

,column_name,discovered_datatypes,num_discovered_datatypes,files_seen
0,Average Road Speed,[double],1,39
1,Average Travel Time,[double],1,39
2,Borough,[string],1,39
3,Bus Trip Count,[int64],1,39
4,Day of Week,[string],1,39
5,Direction,[string],1,39
6,Hour of Day,[int64],1,39
7,Month,[int64],1,39
8,Next Timepoint Stop Georeference,[string],1,39
9,Next Timepoint Stop ID,[int64],1,39


Findings: Schema validation passed with 100% uniformity across all 39 chunks, confirming that all temporal, categorical, and numeric metric columns perfectly align with official NYS Open Data specifications without any datatype conflicts\.

In [13]:
# ---------------------------------------------------------------------
# Check for nulls and numeric boundaries across parquet files
# Uses cached QA manifest unless REBUILD_QA = True
# ---------------------------------------------------------------------

BUS_NULL_BOUNDARY_QA_MANIFEST = QA_DIR / "bus_null_boundary_qa.parquet"

if BUS_NULL_BOUNDARY_QA_MANIFEST.exists() and not REBUILD_QA:
    null_boundary_qa_df = pd.read_parquet(BUS_NULL_BOUNDARY_QA_MANIFEST)
    print(f"Loaded cached null/boundary QA: {BUS_NULL_BOUNDARY_QA_MANIFEST}")

else:
    parquet_chunks = sorted(BUS_PARQUET_DIR.glob("*.parquet"))

    qa_records = []

    print("Scanning chunks for missing values and numeric boundaries...")

    for chunk_file in parquet_chunks:
        df = pd.read_parquet(chunk_file)
        row_count = len(df)

        null_sums = df.isnull().sum()

        numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

        for col in df.columns:
            record = {
                "file": chunk_file.name,
                "column_name": col,
                "row_count": row_count,
                "missing_values": int(null_sums[col]),
                "is_numeric": col in numeric_cols,
                "min_value": None,
                "max_value": None,
            }

            if col in numeric_cols:
                record["min_value"] = df[col].min()
                record["max_value"] = df[col].max()

            qa_records.append(record)

        del df

    null_boundary_qa_df = pd.DataFrame(qa_records)

    null_boundary_qa_df.to_parquet(
        BUS_NULL_BOUNDARY_QA_MANIFEST,
        index=False
    )

    print(f"Rebuilt null/boundary QA: {BUS_NULL_BOUNDARY_QA_MANIFEST}")

# Summarize cached or rebuilt results
total_global_rows = (
    null_boundary_qa_df
    .drop_duplicates("file")["row_count"]
    .sum()
)

missing_summary_df = (
    null_boundary_qa_df
    .groupby("column_name", as_index=False)
    .agg(
        missing_values=("missing_values", "sum"),
        files_seen=("file", "nunique"),
    )
)

missing_summary_df["missing_rate"] = (
    missing_summary_df["missing_values"] / total_global_rows
)

numeric_boundary_df = (
    null_boundary_qa_df[null_boundary_qa_df["is_numeric"]]
    .groupby("column_name", as_index=False)
    .agg(
        min_value=("min_value", "min"),
        max_value=("max_value", "max"),
        files_seen=("file", "nunique"),
    )
)

print(f"\n✅ Total Rows Processed: {total_global_rows:,}\n")

print("=== 🟥 MISSING VALUE REPORT ===")
for _, row in missing_summary_df.iterrows():
    col = row["column_name"]
    count = int(row["missing_values"])
    rate = row["missing_rate"]

    if count > 0:
        print(f"  ⚠️ {col:<32} | {count:,} missing values ({rate:.4%})")
    else:
        print(f"  ✅ {col:<32} | 0 missing values")

print("\n=== 🔢 MIN / MAX BOUNDARY REPORT (WEIRD VALUE CHECK) ===")
for _, row in numeric_boundary_df.iterrows():
    print(
        f"  • {row['column_name']:<32} "
        f"-> Min: {row['min_value']:<10} | Max: {row['max_value']}"
    )

display(missing_summary_df)
display(numeric_boundary_df)

Loaded cached null/boundary QA: pipeline_data/qa/bus_null_boundary_qa.parquet

✅ Total Rows Processed: 18,937,024

=== 🟥 MISSING VALUE REPORT ===
  ✅ Average Road Speed               | 0 missing values
  ✅ Average Travel Time              | 0 missing values
  ✅ Borough                          | 0 missing values
  ✅ Bus Trip Count                   | 0 missing values
  ✅ Day of Week                      | 0 missing values
  ✅ Direction                        | 0 missing values
  ✅ Hour of Day                      | 0 missing values
  ✅ Month                            | 0 missing values
  ⚠️ Next Timepoint Stop Georeference | 10,902 missing values (0.0576%)
  ✅ Next Timepoint Stop ID           | 0 missing values
  ⚠️ Next Timepoint Stop Latitude     | 10,902 missing values (0.0576%)
  ⚠️ Next Timepoint Stop Longitude    | 10,902 missing values (0.0576%)
  ⚠️ Next Timepoint Stop Name         | 10,902 missing values (0.0576%)
  ✅ Road Distance                    | 0 missing values
  ✅ Ro

,column_name,missing_values,files_seen,missing_rate
0,Average Road Speed,0,39,0.000000
1,Average Travel Time,0,39,0.000000
2,Borough,0,39,0.000000
3,Bus Trip Count,0,39,0.000000
4,Day of Week,0,39,0.000000
5,Direction,0,39,0.000000
6,Hour of Day,0,39,0.000000
7,Month,0,39,0.000000
8,Next Timepoint Stop Georeference,10902,39,0.000576
9,Next Timepoint Stop ID,0,39,0.000000


,column_name,min_value,max_value,files_seen
0,Average Road Speed,0.000000,39.999984,39
1,Average Travel Time,0.005562,89.966640,39
2,Bus Trip Count,1.000000,112.000000,39
3,Hour of Day,0.000000,23.000000,39
4,Month,1.000000,12.000000,39
5,Next Timepoint Stop ID,100014.000000,992080.000000,39
6,Next Timepoint Stop Latitude,40.502981,40.933637,39
7,Next Timepoint Stop Longitude,-74.251470,-73.701230,39
8,Road Distance,0.000000,26.126000,39
9,Stop Order,1.000000,124.000000,39


Findings\. Core temporal, route, direction, borough, roadway, speed, travel\-time, and trip\-count fields have no missing values across 18\.9 million observations\. Missing values are limited to a small share of stop\-name, latitude/longitude, and georeference fields, affecting roughly 10\.9K rows\. Numeric boundaries generally look reasonable for NYC bus movement, although zero road\-distance and zero\-speed records should be inspected later as potential edge cases\.

### Candidate Key Identification 

Before performing duplicate\-record checks, we first need to determine the appropriate observation grain for the bus dataset\. Multiple records may legitimately share the same timestamp, route, or stop order while representing different directions, stop pairs, route variants, or roadway segments\. To better understand the dataset's natural grain, we evaluate several candidate keys and assess the frequency of repeated observations under each definition before attempting to identify true duplicates\.

In [14]:
# ---------------------------------------------------------------------
# Test candidate unique keys for bus route-segment observations
# ---------------------------------------------------------------------
# Bus speeds are profile-style route-segment observations, not simple daily
# records. We test progressively richer candidate keys to learn the natural
# grain before labeling repeated key combinations as duplicates.

BUS_DUPLICATE_QA_MANIFEST = QA_DIR / "bus_candidate_key_qa.parquet"

candidate_keys = {
    "timestamp_route_stop_order": [
        "Timestamp",
        "Route ID",
        "Stop Order",
    ],
    "timestamp_route_direction_stop_pair": [
        "Timestamp",
        "Route ID",
        "Direction",
        "Timepoint Stop ID",
        "Next Timepoint Stop ID",
    ],
    "timestamp_route_direction_stop_order_stop_pair": [
        "Timestamp",
        "Route ID",
        "Direction",
        "Stop Order",
        "Timepoint Stop ID",
        "Next Timepoint Stop ID",
    ],
    "timestamp_route_type_direction_stop_order_stop_pair": [
        "Timestamp",
        "Route ID",
        "Route Type",
        "Direction",
        "Stop Order",
        "Timepoint Stop ID",
        "Next Timepoint Stop ID",
    ],
}

if BUS_DUPLICATE_QA_MANIFEST.exists() and not REBUILD_QA:
    candidate_key_qa_df = pd.read_parquet(BUS_DUPLICATE_QA_MANIFEST)
    print(f"Loaded cached candidate key QA: {BUS_DUPLICATE_QA_MANIFEST}")

else:
    parquet_chunks = sorted(BUS_PARQUET_DIR.glob("*.parquet"))
    results = []

    for key_name, key_cols in candidate_keys.items():
        print(f"Testing candidate key: {key_name}")

        key_counts = []

        for chunk_file in parquet_chunks:
            df_key = pd.read_parquet(chunk_file, columns=key_cols)

            chunk_counts = (
                df_key
                .value_counts(dropna=False)
                .reset_index(name="count")
            )

            key_counts.append(chunk_counts)
            del df_key

        combined_counts = pd.concat(key_counts, ignore_index=True)

        global_counts = (
            combined_counts
            .groupby(key_cols, dropna=False, as_index=False)["count"]
            .sum()
        )

        total_rows = global_counts["count"].sum()
        unique_keys = len(global_counts)
        duplicate_rows = total_rows - unique_keys
        duplicate_key_count = (global_counts["count"] > 1).sum()
        max_records_per_key = global_counts["count"].max()

        results.append({
            "candidate_key": key_name,
            "key_columns": ", ".join(key_cols),
            "total_rows": total_rows,
            "unique_keys": unique_keys,
            "duplicate_rows": duplicate_rows,
            "duplicate_key_count": duplicate_key_count,
            "duplicate_row_rate": duplicate_rows / total_rows,
            "max_records_per_key": max_records_per_key,
        })

    candidate_key_qa_df = pd.DataFrame(results)
    candidate_key_qa_df.to_parquet(BUS_DUPLICATE_QA_MANIFEST, index=False)

    print(f"Rebuilt candidate key QA: {BUS_DUPLICATE_QA_MANIFEST}")

candidate_key_qa_df

Loaded cached candidate key QA: pipeline_data/qa/bus_candidate_key_qa.parquet


,candidate_key,key_columns,total_rows,unique_keys,duplicate_rows,duplicate_key_count,duplicate_row_rate,max_records_per_key
0,timestamp_route_stop_order,"Timestamp, Route ID, Stop Order",18937024,2388190,16548834,2351750,0.873888,63
1,timestamp_route_direction_stop_pair,"Timestamp, Route ID, Direction, Timepoint Stop...",18937024,2736808,16200216,2704051,0.855478,42
2,timestamp_route_direction_stop_order_stop_pair,"Timestamp, Route ID, Direction, Stop Order, Ti...",18937024,2812128,16124896,2753092,0.851501,33
3,timestamp_route_type_direction_stop_order_stop...,"Timestamp, Route ID, Route Type, Direction, St...",18937024,2925980,16011044,2859450,0.845489,20


Findings\. None of the candidate keys performed particularly well, with duplicate rates remaining above 84% even after progressively adding route, direction, stop\-pair, and route\-type information\. This suggests that the dataset contains an additional dimension not captured by the tested keys\. Rather than indicating a widespread duplicate\-record problem, the results suggest that our understanding of the dataset's observation grain is incomplete and warrants closer inspection of the repeated records\.

In [15]:
# ---------------------------------------------------------------------
# Inspect repeated candidate keys vs. exact duplicate rows
# ---------------------------------------------------------------------

key_cols = [
    "Timestamp",
    "Route ID",
    "Route Type",
    "Direction",
    "Stop Order",
    "Timepoint Stop ID",
    "Next Timepoint Stop ID",
]

sample_repeated_groups = []
exact_duplicate_summary = []

for path in sorted(BUS_PARQUET_DIR.glob("*.parquet")):
    df = pd.read_parquet(path)

    total_rows = len(df)
    exact_duplicate_rows = df.duplicated(keep="first").sum()

    exact_duplicate_summary.append({
        "file": path.name,
        "total_rows": total_rows,
        "exact_duplicate_rows": exact_duplicate_rows,
        "exact_duplicate_rate": exact_duplicate_rows / total_rows if total_rows else 0,
    })

    repeated = df[df.duplicated(subset=key_cols, keep=False)]

    if not repeated.empty:
        sample_repeated_groups.append(
            repeated
            .sort_values(key_cols)
            .head(50)
        )

    del df

    if sample_repeated_groups:
        break

exact_duplicate_summary_df = pd.DataFrame(exact_duplicate_summary)

print("Exact duplicate summary from scanned partition(s):")
display(exact_duplicate_summary_df)

if sample_repeated_groups:
    repeated_key_sample_df = pd.concat(sample_repeated_groups, ignore_index=True)

    print("Sample rows sharing the strongest candidate key:")
    display(repeated_key_sample_df)
else:
    print("No repeated candidate keys found in scanned partitions.")

Exact duplicate summary from scanned partition(s):


,file,total_rows,exact_duplicate_rows,exact_duplicate_rate
0,bus_2023_2024_part_0000.parquet,500000,0,0.0


Sample rows sharing the strongest candidate key:


,Year,Month,Timestamp,Day of Week,Hour of Day,Route ID,Direction,Borough,Route Type,Stop Order,Timepoint Stop ID,Timepoint Stop Name,Timepoint Stop Latitude,Timepoint Stop Longitude,Next Timepoint Stop ID,Next Timepoint Stop Name,Next Timepoint Stop Latitude,Next Timepoint Stop Longitude,Road Distance,Average Travel Time,Average Road Speed,Bus Trip Count,Timepoint Stop Georeference,Next Timepoint Stop Georeference
0,2023,1,2023-01-01,Thursday,0,B12,E,Brooklyn,Local,19,306477,EMPIRE BL/UTICA AV,40.663496,-73.931681,300709,SARATOGA AV/PITKIN AV,40.669033,-73.917339,0.861,7.146972,7.228237,11,POINT (-73.931681 40.663496),POINT (-73.917339 40.669033)
1,2023,1,2023-01-01,Friday,0,B12,E,Brooklyn,Local,19,306477,EMPIRE BL/UTICA AV,40.663496,-73.931681,300709,SARATOGA AV/PITKIN AV,40.669033,-73.917339,0.861,6.651384,7.766803,12,POINT (-73.931681 40.663496),POINT (-73.917339 40.669033)
2,2023,1,2023-01-01,Wednesday,0,B12,E,Brooklyn,Local,19,306477,EMPIRE BL/UTICA AV,40.663496,-73.931681,300709,SARATOGA AV/PITKIN AV,40.669033,-73.917339,0.861,6.827778,7.566149,12,POINT (-73.931681 40.663496),POINT (-73.917339 40.669033)
3,2023,1,2023-01-01,Tuesday,0,B12,E,Brooklyn,Local,19,306477,EMPIRE BL/UTICA AV,40.663496,-73.931681,300709,SARATOGA AV/PITKIN AV,40.669033,-73.917339,0.861,6.574446,7.857699,15,POINT (-73.931681 40.663496),POINT (-73.917339 40.669033)
4,2023,1,2023-01-01,Monday,0,B12,E,Brooklyn,Local,19,306477,EMPIRE BL/UTICA AV,40.663496,-73.931681,300709,SARATOGA AV/PITKIN AV,40.669033,-73.917339,0.861,6.658332,7.758699,14,POINT (-73.931681 40.663496),POINT (-73.917339 40.669033)
5,2023,1,2023-01-01,Saturday,0,B12,E,Brooklyn,Local,19,306477,EMPIRE BL/UTICA AV,40.663496,-73.931681,300709,SARATOGA AV/PITKIN AV,40.669033,-73.917339,0.861,6.548604,7.888703,12,POINT (-73.931681 40.663496),POINT (-73.917339 40.669033)
6,2023,1,2023-01-01,Sunday,0,B12,E,Brooklyn,Local,19,306477,EMPIRE BL/UTICA AV,40.663496,-73.931681,300709,SARATOGA AV/PITKIN AV,40.669033,-73.917339,0.861,7.159524,7.215562,7,POINT (-73.931681 40.663496),POINT (-73.917339 40.669033)
7,2023,1,2023-01-01,Thursday,0,B13,N,Brooklyn,Local,54,504119,GATES AV/WYCKOFF AV,40.700050,-73.911769,901269,WYCKOFF AV/DE KALB AV,40.704385,-73.919467,0.782,4.707138,9.967842,7,POINT (-73.911769 40.70005),POINT (-73.919467 40.704385)
8,2023,1,2023-01-01,Friday,0,B13,N,Brooklyn,Local,54,504119,GATES AV/WYCKOFF AV,40.700050,-73.911769,901269,WYCKOFF AV/DE KALB AV,40.704385,-73.919467,0.782,4.820838,9.732754,8,POINT (-73.911769 40.70005),POINT (-73.919467 40.704385)
9,2023,1,2023-01-01,Wednesday,0,B13,N,Brooklyn,Local,54,504119,GATES AV/WYCKOFF AV,40.700050,-73.911769,901269,WYCKOFF AV/DE KALB AV,40.704385,-73.919467,0.782,3.945840,11.891004,8,POINT (-73.911769 40.70005),POINT (-73.919467 40.704385)


Findings\. Inspection of the repeated records revealed the missing piece\. Observations that appeared identical across route, segment, and time fields were often distinguished by Day of Week, with separate mobility measurements reported for Monday through Sunday\. This explains why the candidate\-key analysis produced such high duplicate rates and suggests that Day of Week is part of the dataset's natural observation grain\.

Let's try this new key and see if we still have a duplication problem

In [16]:
# ---------------------------------------------------------------------
# QA duplicate records using revised bus observation grain
# ---------------------------------------------------------------------
# The duplicate investigation showed that Timestamp is not the true date-level
# grain for this source. Bus speeds are reported as recurring route-segment
# profiles by year, month, day of week, hour, and route/segment identifiers.
# This revised key checks duplicates against that source-specific structure.

bus_observation_key = [
    "Year",
    "Month",
    "Day of Week",
    "Hour of Day",
    "Route ID",
    "Route Type",
    "Direction",
    "Stop Order",
    "Timepoint Stop ID",
    "Next Timepoint Stop ID",
]

BUS_REVISED_DUPLICATE_QA_MANIFEST = QA_DIR / "bus_revised_duplicate_qa.parquet"
BUS_REVISED_DUPLICATE_SAMPLE = QA_DIR / "bus_revised_duplicate_sample.parquet"

if BUS_REVISED_DUPLICATE_QA_MANIFEST.exists() and not REBUILD_QA:
    revised_duplicate_qa_df = pd.read_parquet(BUS_REVISED_DUPLICATE_QA_MANIFEST)
    print(f"Loaded cached duplicate QA: {BUS_REVISED_DUPLICATE_QA_MANIFEST}")

    if BUS_REVISED_DUPLICATE_SAMPLE.exists():
        revised_duplicate_sample_df = pd.read_parquet(BUS_REVISED_DUPLICATE_SAMPLE)
    else:
        revised_duplicate_sample_df = pd.DataFrame()

else:
    duplicate_summary = []
    duplicate_samples = []

    for path in sorted(BUS_PARQUET_DIR.glob("*.parquet")):
        df = pd.read_parquet(path)

        total_rows = len(df)

        exact_duplicate_rows = df.duplicated(keep="first").sum()
        key_duplicate_rows = df.duplicated(subset=bus_observation_key, keep="first").sum()
        key_duplicate_groups = df[df.duplicated(subset=bus_observation_key, keep=False)]

        duplicate_summary.append({
            "file": path.name,
            "total_rows": total_rows,
            "exact_duplicate_rows": exact_duplicate_rows,
            "exact_duplicate_rate": exact_duplicate_rows / total_rows if total_rows else 0,
            "key_duplicate_rows": key_duplicate_rows,
            "key_duplicate_rate": key_duplicate_rows / total_rows if total_rows else 0,
            "key_duplicate_group_rows": len(key_duplicate_groups),
        })

        if not key_duplicate_groups.empty and len(duplicate_samples) < 3:
            duplicate_samples.append(
                key_duplicate_groups
                .sort_values(bus_observation_key)
                .head(50)
            )

        del df

    revised_duplicate_qa_df = pd.DataFrame(duplicate_summary)

    revised_duplicate_qa_df.to_parquet(
        BUS_REVISED_DUPLICATE_QA_MANIFEST,
        index=False
    )

    if duplicate_samples:
        revised_duplicate_sample_df = pd.concat(duplicate_samples, ignore_index=True)
        revised_duplicate_sample_df.to_parquet(
            BUS_REVISED_DUPLICATE_SAMPLE,
            index=False
        )
    else:
        revised_duplicate_sample_df = pd.DataFrame()

    print(f"Rebuilt duplicate QA: {BUS_REVISED_DUPLICATE_QA_MANIFEST}")

summary_totals = pd.DataFrame([{
    "files_checked": len(revised_duplicate_qa_df),
    "total_rows": revised_duplicate_qa_df["total_rows"].sum(),
    "exact_duplicate_rows": revised_duplicate_qa_df["exact_duplicate_rows"].sum(),
    "exact_duplicate_rate": (
        revised_duplicate_qa_df["exact_duplicate_rows"].sum()
        / revised_duplicate_qa_df["total_rows"].sum()
    ),
    "key_duplicate_rows": revised_duplicate_qa_df["key_duplicate_rows"].sum(),
    "key_duplicate_rate": (
        revised_duplicate_qa_df["key_duplicate_rows"].sum()
        / revised_duplicate_qa_df["total_rows"].sum()
    ),
    "key_duplicate_group_rows": revised_duplicate_qa_df["key_duplicate_group_rows"].sum(),
}])

print("Duplicate QA summary using revised observation key:")
display(summary_totals)

print("Partition-level duplicate QA:")
display(revised_duplicate_qa_df)

if not revised_duplicate_sample_df.empty:
    print("Sample rows still sharing the revised observation key:")
    display(revised_duplicate_sample_df)
else:
    print("No repeated rows found using the revised observation key.")

Loaded cached duplicate QA: pipeline_data/qa/bus_revised_duplicate_qa.parquet
Duplicate QA summary using revised observation key:


,files_checked,total_rows,exact_duplicate_rows,exact_duplicate_rate,key_duplicate_rows,key_duplicate_rate,key_duplicate_group_rows
0,39,18937024,1,5.280661e-08,263888,0.013935,526680


Partition-level duplicate QA:


,file,total_rows,exact_duplicate_rows,exact_duplicate_rate,key_duplicate_rows,key_duplicate_rate,key_duplicate_group_rows
0,bus_2023_2024_part_0000.parquet,500000,0,0.000000,6122,0.012244,12208
1,bus_2023_2024_part_0001.parquet,500000,0,0.000000,5276,0.010552,10539
2,bus_2023_2024_part_0002.parquet,500000,0,0.000000,6886,0.013772,13759
3,bus_2023_2024_part_0003.parquet,500000,0,0.000000,7455,0.014910,14828
4,bus_2023_2024_part_0004.parquet,500000,0,0.000000,7124,0.014248,14235
5,bus_2023_2024_part_0005.parquet,500000,0,0.000000,6433,0.012866,12835
6,bus_2023_2024_part_0006.parquet,500000,0,0.000000,6806,0.013612,13583
7,bus_2023_2024_part_0007.parquet,500000,0,0.000000,5842,0.011684,11665
8,bus_2023_2024_part_0008.parquet,500000,0,0.000000,4504,0.009008,8999
9,bus_2023_2024_part_0009.parquet,500000,0,0.000000,10797,0.021594,21565


Sample rows still sharing the revised observation key:


,Year,Month,Timestamp,Day of Week,Hour of Day,Route ID,Direction,Borough,Route Type,Stop Order,Timepoint Stop ID,Timepoint Stop Name,Timepoint Stop Latitude,Timepoint Stop Longitude,Next Timepoint Stop ID,Next Timepoint Stop Name,Next Timepoint Stop Latitude,Next Timepoint Stop Longitude,Road Distance,Average Travel Time,Average Road Speed,Bus Trip Count,Timepoint Stop Georeference,Next Timepoint Stop Georeference
0,2023,1,2023-01-01 00:00:00,Friday,0,M125,W,Manhattan,Local,18,403604,W 125 ST/AMSTERDAM AV,40.813693,-73.956392,403829,W 125 ST/ST CLAIRE PL,40.817202,-73.959504,0.292,3.893754,4.499516,8,POINT (-73.956392 40.813693),POINT (-73.959504 40.817202)
1,2023,1,2023-01-01 00:00:00,Friday,0,M125,W,Manhattan,Local,18,403604,W 125 ST/AMSTERDAM AV,40.813693,-73.956392,403829,W 125 ST/ST CLAIRE PL,40.817202,-73.959504,0.292,2.283342,7.672970,3,POINT (-73.956392 40.813693),POINT (-73.959504 40.817202)
2,2023,1,2023-01-01 00:00:00,Friday,0,Q54,W,Queens,Local,1,504384,JAMAICA AV/170 ST,40.707288,-73.789580,503718,JAMAICA AV/SUTPHIN BL,40.701861,-73.808767,1.039,5.977086,10.429829,8,POINT (-73.78958 40.707288),POINT (-73.808767 40.701861)
3,2023,1,2023-01-01 00:00:00,Friday,0,Q54,W,Queens,Local,1,504384,JAMAICA AV/170 ST,40.707288,-73.789580,503718,JAMAICA AV/SUTPHIN BL,40.701861,-73.808767,1.632,5.966670,16.411164,2,POINT (-73.78958 40.707288),POINT (-73.808767 40.701861)
4,2023,1,2023-01-01 00:00:00,Friday,0,Q72,S,Queens,Local,3,551131,94 ST/DITMARS BL,40.770078,-73.876107,552020,94 ST/ASTORIA BL,40.762602,-73.875288,0.523,2.830548,11.086185,6,POINT (-73.876107 40.770078),POINT (-73.875288 40.762602)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,2023,1,2023-01-01 08:00:00,Friday,8,S53,E,Staten Island,Local,1,202740,RICHMOND TER/PARK AV,40.640046,-74.130627,200261,CASTLETON AV/JEWETT AV,40.633639,-74.129649,0.737,6.075000,7.279012,6,POINT (-74.130627 40.640046),POINT (-74.129649 40.633639)
146,2023,1,2023-01-01 08:00:00,Friday,8,S57,S,Staten Island,Local,1,202740,RICHMOND TER/PARK AV,40.640046,-74.130627,200428,PORT RICHMOND AV/CHARLES AV,40.634730,-74.135754,0.519,2.751846,11.316037,9,POINT (-74.130627 40.640046),POINT (-74.135754 40.63473)
147,2023,1,2023-01-01 08:00:00,Friday,8,S57,S,Staten Island,Local,1,202740,RICHMOND TER/PARK AV,40.640046,-74.130627,200428,PORT RICHMOND AV/CHARLES AV,40.634730,-74.135754,0.525,4.000020,7.874961,1,POINT (-74.130627 40.640046),POINT (-74.135754 40.63473)
148,2023,1,2023-01-01 09:00:00,Friday,9,B103,E,Brooklyn,Limited,24,350101,E 80 ST / FLATLANDS AV,40.635351,-73.913170,350243,WILLIAMS AV / FLATLANDS AV,40.650381,-73.891446,2.662,16.130946,9.899869,7,POINT (-73.91317 40.635351),POINT (-73.891446 40.650381)


Findings\. After defining a key that does NOT use Timestamp, the staged bus dataset does not appear to have a meaningful duplicate\-record problem\. After refining the observation key to include Day of Week and Hour of Day, only 1\.4% of records remained duplicated and just a single exact duplicate row was identified across 18\.9 million observations\.

Key Learnings\. The staged bus dataset passed all major QA checks and appears ready for downstream analysis\. Missingness and exact duplication were both minimal, schema consistency was maintained across all parquet partitions, and the duplicate investigation clarified that the dataset is organized around Year, Month, Day of Week, Hour of Day, and route\-segment characteristics rather than unique timestamps\. These findings provide a clear foundation for temporal standardization and multimodal integration\.

## 1\.1\.4\.5 Validate categorical distributions

Let's now check counts based on our nominal and categorical variables\.

In [17]:
# ---------------------------------------------------------------------
# Check categorical distributions across parquet files
# Uses cached QA manifest unless REBUILD_QA = True
# ---------------------------------------------------------------------

BUS_CATEGORICAL_QA_MANIFEST = QA_DIR / "bus_categorical_distribution_qa.parquet"

categorical_cols = ["Borough", "Route Type", "Direction"]

if BUS_CATEGORICAL_QA_MANIFEST.exists() and not REBUILD_QA:
    categorical_qa_df = pd.read_parquet(BUS_CATEGORICAL_QA_MANIFEST)
    print(f"Loaded cached categorical QA: {BUS_CATEGORICAL_QA_MANIFEST}")

else:
    parquet_chunks = sorted(BUS_PARQUET_DIR.glob("*.parquet"))
    categorical_records = []

    print("Calculating categorical distributions...")

    for chunk_file in parquet_chunks:
        df_cat = pd.read_parquet(chunk_file, columns=categorical_cols)

        for col in categorical_cols:
            counts = df_cat[col].value_counts(dropna=False)

            for value, count in counts.items():
                categorical_records.append({
                    "file": chunk_file.name,
                    "column_name": col,
                    "value": value,
                    "row_count": int(count),
                })

        del df_cat

    categorical_qa_df = pd.DataFrame(categorical_records)

    categorical_qa_df.to_parquet(
        BUS_CATEGORICAL_QA_MANIFEST,
        index=False
    )

    print(f"Rebuilt categorical QA: {BUS_CATEGORICAL_QA_MANIFEST}")

categorical_summary_df = (
    categorical_qa_df
    .groupby(["column_name", "value"], dropna=False, as_index=False)
    .agg(row_count=("row_count", "sum"))
)

total_rows = (
    categorical_summary_df[categorical_summary_df["column_name"] == "Borough"]
    ["row_count"]
    .sum()
)

categorical_summary_df["row_share"] = (
    categorical_summary_df["row_count"] / total_rows
)

for col in categorical_cols:
    print(f"\n=== {col.upper()} VALUE COUNTS ===")

    col_summary = (
        categorical_summary_df[categorical_summary_df["column_name"] == col]
        .sort_values("row_count", ascending=False)
    )

    for _, row in col_summary.iterrows():
        value = row["value"]
        count = row["row_count"]
        share = row["row_share"]

        print(f"  {str(value):<20}: {count:,} ({share:.2%})")

display(categorical_summary_df)

Loaded cached categorical QA: pipeline_data/qa/bus_categorical_distribution_qa.parquet

=== BOROUGH VALUE COUNTS ===
  Queens              : 5,537,719 (29.24%)
  Brooklyn            : 4,785,110 (25.27%)
  Bronx               : 3,473,047 (18.34%)
  Manhattan           : 2,627,748 (13.88%)
  Staten Island       : 1,845,199 (9.74%)
  Other               : 668,201 (3.53%)

=== ROUTE TYPE VALUE COUNTS ===
  Local               : 15,195,139 (80.24%)
  Express             : 1,887,736 (9.97%)
  Limited             : 1,046,871 (5.53%)
  SBS                 : 626,339 (3.31%)
  School              : 177,494 (0.94%)
  School Limited      : 3,445 (0.02%)

=== DIRECTION VALUE COUNTS ===
  N                   : 5,052,527 (26.68%)
  S                   : 4,943,335 (26.10%)
  W                   : 4,518,990 (23.86%)
  E                   : 4,422,172 (23.35%)


,column_name,value,row_count,row_share
0,Borough,Bronx,3473047,0.183400
1,Borough,Brooklyn,4785110,0.252685
2,Borough,Manhattan,2627748,0.138762
3,Borough,Other,668201,0.035285
4,Borough,Queens,5537719,0.292428
5,Borough,Staten Island,1845199,0.097439
6,Direction,E,4422172,0.233520
7,Direction,N,5052527,0.266807
8,Direction,S,4943335,0.261041
9,Direction,W,4518990,0.238633


Findings\. Categorical coverage looks sensible\. Observations span all five boroughs plus an “Other” category, with Queens and Brooklyn representing the largest shares of bus segment observations\. Most records are Local routes, with smaller volumes for SBS, Limited, Express, and School service\. Direction values are also cleanly distributed across N, S, E, and W, with no unexpected direction labels detected\.

## 1\.1\.4\.6 Bus mobility metric validation

Before using bus speed, travel time, trip count, and roadway distance measurements in downstream analysis, we first validate that these metrics contain reasonable values and distributions\. The goal of this section is not to perform exploratory analysis, but rather to identify potential data\-quality issues such as impossible values, extreme outliers, or suspicious measurement patterns that may require cleaning or special handling during later stages of the pipeline\.

In [18]:
# ---------------------------------------------------------------------
# Validate bus mobility metric distributions
# Uses cached QA manifest unless REBUILD_QA = True
# ---------------------------------------------------------------------

BUS_METRIC_QA_MANIFEST = QA_DIR / "bus_metric_distribution_qa.parquet"

metric_cols = [
    "Average Road Speed",
    "Average Travel Time",
    "Bus Trip Count",
    "Road Distance",
]

if BUS_METRIC_QA_MANIFEST.exists() and not REBUILD_QA:
    metric_qa_df = pd.read_parquet(BUS_METRIC_QA_MANIFEST)
    print(f"Loaded cached metric QA: {BUS_METRIC_QA_MANIFEST}")

else:
    parquet_chunks = sorted(BUS_PARQUET_DIR.glob("*.parquet"))
    metric_records = []

    print("Scanning bus mobility metrics across parquet chunks...")

    for chunk_file in parquet_chunks:
        df_metrics = pd.read_parquet(chunk_file, columns=metric_cols)

        for col in metric_cols:
            series = df_metrics[col].dropna()

            metric_records.append({
                "file": chunk_file.name,
                "metric": col,
                "row_count": len(df_metrics),
                "non_null_count": series.count(),
                "missing_count": df_metrics[col].isna().sum(),
                "min": series.min(),
                "p01": series.quantile(0.01),
                "p05": series.quantile(0.05),
                "p25": series.quantile(0.25),
                "median": series.quantile(0.50),
                "mean": series.mean(),
                "p75": series.quantile(0.75),
                "p95": series.quantile(0.95),
                "p99": series.quantile(0.99),
                "max": series.max(),
                "zero_count": (series == 0).sum(),
                "negative_count": (series < 0).sum(),
            })

        del df_metrics

    metric_qa_df = pd.DataFrame(metric_records)

    metric_qa_df.to_parquet(
        BUS_METRIC_QA_MANIFEST,
        index=False
    )

    print(f"Rebuilt metric QA: {BUS_METRIC_QA_MANIFEST}")

# ---------------------------------------------------------------------
# Summarize across all chunks
# ---------------------------------------------------------------------

metric_summary_df = (
    metric_qa_df
    .groupby("metric", as_index=False)
    .agg(
        total_rows=("row_count", "sum"),
        non_null_count=("non_null_count", "sum"),
        missing_count=("missing_count", "sum"),
        min=("min", "min"),
        p01=("p01", "median"),
        p05=("p05", "median"),
        p25=("p25", "median"),
        median=("median", "median"),
        mean=("mean", "mean"),
        p75=("p75", "median"),
        p95=("p95", "median"),
        p99=("p99", "median"),
        max=("max", "max"),
        zero_count=("zero_count", "sum"),
        negative_count=("negative_count", "sum"),
    )
)

metric_summary_df["missing_rate"] = (
    metric_summary_df["missing_count"] / metric_summary_df["total_rows"]
)

metric_summary_df["zero_rate"] = (
    metric_summary_df["zero_count"] / metric_summary_df["non_null_count"]
)

metric_summary_df["negative_rate"] = (
    metric_summary_df["negative_count"] / metric_summary_df["non_null_count"]
)

display(metric_summary_df)

Loaded cached metric QA: pipeline_data/qa/bus_metric_distribution_qa.parquet


,metric,total_rows,non_null_count,missing_count,min,p01,p05,p25,median,mean,p75,p95,p99,max,zero_count,negative_count,missing_rate,zero_rate,negative_rate
0,Average Road Speed,18937024,18937024,0,0.000000,3.456095,4.573998,6.389606,8.167828,9.013198,10.623704,16.050531,22.423538,39.999984,57,0,0.0,0.000003,0.0
1,Average Travel Time,18937024,18937024,0,0.005562,1.517535,2.656254,5.283330,7.900008,9.184037,11.445834,19.813684,32.920013,89.966640,0,0,0.0,0.000000,0.0
2,Bus Trip Count,18937024,18937024,0,1.000000,2.000000,3.000000,8.000000,12.000000,14.473364,20.000000,32.000000,43.000000,112.000000,0,0,0.0,0.000000,0.0
3,Road Distance,18937024,18937024,0,0.000000,0.180000,0.332000,0.714000,1.079000,1.377625,1.529000,3.124000,7.299000,26.126000,57,0,0.0,0.000003,0.0


Findings\. Core bus mobility metrics look usable for downstream analysis\. Average road speed, travel time, trip count, and road distance all have complete coverage across 18\.9 million observations, with no missing or negative values\. A very small number of zero\-speed and zero\-distance records were detected, but they represent only 57 rows and do not appear material at this stage\. Metric ranges are broadly plausible for NYC bus operations, so no corrective action is needed before moving into later pipeline stages\.

## Conclusion

The staged bus dataset passed all major validation checks and appears ready for downstream processing\. The parquet conversion preserved the full schema and temporal information, missingness and exact duplication were minimal, and mobility metrics exhibited reasonable value ranges\. The most important finding from this inspection was that the dataset is organized around Year, Month, Day of Week, and Hour of Day rather than unique timestamps, a distinction that will inform temporal standardization and multimodal integration in later stages of the project\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>